# Step 1 — Smoke test

**Gate: do not build anything past this notebook until every cell below passes.**

Tests, in order: (1) Phi-3-vision loads with `attn_implementation="eager"` + 4-bit
quantization — this is the corrected config, not the paper's `flash_attention_2`,
since flash attention has no Turing/T4 support and `sdpa` is explicitly unsupported
by this model's custom modeling code (`_supports_sdpa = False`); (2) both LoRA
adapters attach and hot-swap cleanly via PEFT on this `trust_remote_code` custom
class; (3) retrieval and generation produce sane output against NTT's own README
example. Every step is logged to JSONL via `vdocrag.telemetry` so the timing/VRAM
numbers feed directly into the page-cap formula (`vdocrag.limits`) afterward —
this run *is* the data-gathering pass, not a separate benchmark.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/vdocrag-project/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

Mounted at /content/drive


In [2]:
!pip install -q --upgrade pip
# pinned to the last 4.x release before transformers' 5.0 major version --
# 5.x ships a substantially refactored attention-dispatch mechanism that
# breaks Phi-3-vision's custom trust_remote_code modeling file (confirmed
# by actually hitting this in Step 1: unpinned installs grab 5.x and
# 'eager' silently fails with a flash-attention ValueError it never asked for)
!pip install -q transformers==4.57.3 accelerate bitsandbytes peft pillow

# NTT's package is installed by URL every session, never vendored into this
# repo — their license (SOFTWARE LICENSE AGREEMENT FOR EVALUATION, §4(b)(i))
# prohibits redistribution. See docs/licenses.md.
!pip install -q git+https://github.com/nttmdlab-nlp/VDocRAG.git

# clone this repo fresh too, so vdocrag_app.telemetry / vdocrag_app.limits are
# importable (our own package -- see the naming-collision note in the next cell)
# -rf first so re-running this cell mid-troubleshooting doesn't fail on a
# leftover directory from a previous attempt
!rm -rf /content/repo
!git clone https://github.com/thejainamjain/vdocrag-project.git /content/repo
import sys
sys.path.insert(0, '/content/repo')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cloning into '/content/repo'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 102 (delta 31), reused 96 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 85.38 KiB | 1.35 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [3]:
# our own package is named vdocrag_app, NOT vdocrag -- NTT's own
# released package (installed above via pip) is itself named 'vdocrag'
# (see their setup.py: setup(name='vdocrag', ...)). Both can't share the
# name on sys.path -- ours is renamed to vdocrag_app specifically to avoid
# this collision, which otherwise silently shadows one package's submodules.
from vdocrag_app.telemetry import setup_logging, log_call, logger
logger = setup_logging('step1_smoke_test')

Logging initialized: /content/drive/MyDrive/vdocrag-project/logs/step1_smoke_test.jsonl
INFO:vdocrag:Logging initialized: /content/drive/MyDrive/vdocrag-project/logs/step1_smoke_test.jsonl


In [4]:
import transformers
# NTT's vdocretriever.py / vdocgenerator.py import AutoModelForVision2Seq at
# module level, unconditionally. That symbol is a long-deprecated alias for
# AutoModelForImageTextToText and has been removed from transformers ahead of
# the officially documented v5.0 cutoff. NTT's own retriever/generator load via
# AutoModelForCausalLM (their TRANSFORMER_CLS) -- the missing symbol is dead
# code in their file, never actually called.
if not hasattr(transformers, "AutoModelForVision2Seq"):
    transformers.AutoModelForVision2Seq = transformers.AutoModelForImageTextToText
    print("Shimmed AutoModelForVision2Seq -> AutoModelForImageTextToText")

# Phi-3-vision's frozen prepare_inputs_for_generation() reads two Cache
# attributes/methods newer transformers has renamed -- this exact two-part
# breakage is documented across many custom-code models (Phi-3, MiniCPM,
# GOT-OCR, DeepSeek-OCR all hit it identically).
from transformers.cache_utils import DynamicCache

if not hasattr(DynamicCache, "seen_tokens"):
    DynamicCache.seen_tokens = property(lambda self: self.get_seq_length())
    print("Patched DynamicCache.seen_tokens -> get_seq_length()")

if not hasattr(DynamicCache, "get_max_length"):
    DynamicCache.get_max_length = lambda self: self.get_max_cache_shape()
    print("Patched DynamicCache.get_max_length() -> get_max_cache_shape()")

Patched DynamicCache.seen_tokens -> get_seq_length()
Patched DynamicCache.get_max_length() -> get_max_cache_shape()


## Check 1 — base model loads (eager + 4-bit)

**First action before anything else**: open NTT's actual `VDocRetriever.load()` /
`VDocGenerator.load()` signature (in the package just installed, under
`site-packages/vdocrag/` or wherever pip put it — run `!pip show -f vdocrag` to
find it) and confirm whether `attn_implementation` passes through as a kwarg, per
the Tevatron/DSE precedent, or is hardcoded. This determines whether Check 2 below
can call `.load()` directly or needs a wrapper-level override instead.

In [5]:
!pip show -f vdocrag | grep Location
# then manually inspect the retriever/generator load() signature before running Check 2

Location: /usr/local/lib/python3.13/dist-packages


In [6]:
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "microsoft/Phi-3-vision-128k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

@log_call("smoke_test")
def load_base_model():
    # attn_implementation is set directly on the config object, NOT passed as a
    # from_pretrained() kwarg -- confirmed necessary empirically: passing it as a
    # plain kwarg raised `ValueError: Phi3VForCausalLM does not support Flash
    # Attention 2 yet` even though flash attention was never requested. Something
    # in this environment's attention-implementation validation mishandles the
    # kwarg-based route for this custom trust_remote_code model; setting the
    # attribute directly on the config before from_pretrained() bypasses it.
    config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
    config._attn_implementation = "eager"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        config=config,
        trust_remote_code=True,
        device_map="cuda",
        torch_dtype=torch.bfloat16,
        quantization_config=bnb_config,
    )
    return model

base_model = load_base_model()
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
print(f"VRAM after base load: {torch.cuda.memory_allocated()/1e9:.2f}GB, peak: {torch.cuda.max_memory_allocated()/1e9:.2f}GB")
print("CHECK 1: PASS if no exception above and VRAM is in the low single-digit GB range.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

smoke_test.load_base_model ok (187354ms)
INFO:vdocrag:smoke_test.load_base_model ok (187354ms)
/usr/local/lib/python3.13/dist-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


VRAM after base load: 2.44GB, peak: 4.15GB
CHECK 1: PASS if no exception above and VRAM is in the low single-digit GB range.


**If this cell OOMs**: mitigation order per handoff doc Section 6.1 --
(1) reduce max image resolution (paper's own Table G: 672x672 keeps ~72.8 vs
1344x1344's 72.9 nDCG@5 at meaningfully less compute -- a legitimate lever, not
just for speed), (2) reduce batch size (should already be 1 for this demo),
(3) gradient checkpointing is NOT relevant here (inference only, no training).

**If this cell raises a ValueError about attention implementation**: something is
wrong with this notebook, not your environment -- `eager` should never hit the
`_supports_sdpa` gate. Stop and re-check the `attn_implementation` string above.

## Check 2 — LoRA adapters attach and hot-swap

In [7]:
from peft import PeftModel

RETRIEVER_ADAPTER = "NTT-hil-insight/VDocRetriever-Phi3-vision"
GENERATOR_ADAPTER = "NTT-hil-insight/VDocGenerator-Phi3-vision"

@log_call("smoke_test")
def attach_adapters(base_model):
    peft_model = PeftModel.from_pretrained(base_model, RETRIEVER_ADAPTER, adapter_name="retriever")
    peft_model.load_adapter(GENERATOR_ADAPTER, adapter_name="generator")
    return peft_model

peft_model = attach_adapters(base_model)
print(f"VRAM after both adapters attached: {torch.cuda.memory_allocated()/1e9:.2f}GB, peak: {torch.cuda.max_memory_allocated()/1e9:.2f}GB")

import time
t0 = time.time()
peft_model.set_adapter("retriever")
t1 = time.time()
peft_model.set_adapter("generator")
t2 = time.time()
print(f"set_adapter swap timing: retriever={( t1-t0)*1000:.0f}ms, generator={(t2-t1)*1000:.0f}ms")
print("Expect ~50ms per swap per the PEFT hot-swap reference (handoff doc Section 4.4). If this is")
print("much higher, the shared-model VRAM-savings decision may need revisiting for latency reasons.")
print("CHECK 2: PASS if no exception and swap times are in the tens-of-ms range.")

smoke_test.attach_adapters ok (7690ms)
INFO:vdocrag:smoke_test.attach_adapters ok (7690ms)


VRAM after both adapters attached: 2.49GB, peak: 2.50GB
set_adapter swap timing: retriever=9ms, generator=8ms
Expect ~50ms per swap per the PEFT hot-swap reference (handoff doc Section 4.4). If this is
much higher, the shared-model VRAM-savings decision may need revisiting for latency reasons.
CHECK 2: PASS if no exception and swap times are in the tens-of-ms range.


## Check 3 — sane output vs. NTT's actual test.py example

This is NTT's real quickstart example, pulled directly from their repo's `test.py`
(github.com/nttmdlab-nlp/VDocRAG) -- not reconstructed from the paper's prose.
Their README reports similarities of `[0.515625, 0.38476562]` and
`[0.37890625, 0.5703125]` for these two queries against these two images on their
own setup (flash_attention_2, A100). Under `eager` attention + 4-bit quantization
the exact numbers will differ (different attention kernel, different precision),
but the *ordering* should not: query 0 should score higher on image1 than image2,
and vice versa for query 1. If the ordering flips, something is wrong with the
quantization/attention config, not just numerically different as expected.

In [8]:
import torch
from PIL import Image
import requests
from io import BytesIO
from torch.nn.functional import cosine_similarity
from vdocrag.vdocretriever.modeling import VDocRetriever
from vdocrag.vdocgenerator.modeling import VDocGenerator
from vdocrag_app.retriever import build_query_prompt, prepare_doc_image, DOC_PROMPT

torch.cuda.empty_cache()

# num_crops controls actual token count / memory -- NOT image pixel size.
# Confirmed empirically: resizing images did not fix an OOM here, because
# Phi-3-vision's dynamic cropping token count is driven by num_crops (default
# 16, ~2300+ tokens/image under eager attention), not by input pixel dimensions.
# 4 is Microsoft's own suggested value for memory-constrained use -- real
# accuracy trade-off since NTT's checkpoints were fine-tuned against the
# default; revisit once the pipeline runs end-to-end.
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, num_crops=4)

retriever_model = VDocRetriever(encoder=peft_model, pooling="eos", normalize=True)
generator_model = VDocGenerator(decoder=peft_model)

peft_model.set_adapter("retriever")

queries = [
    "What is the total percentage of Palestinians residing at West Bank?",
    "How many international visitors came to Japan in 2017?",
]
query_prompts = [build_query_prompt(q) for q in queries]
query_inputs = processor(query_prompts, return_tensors="pt", padding="longest", max_length=256, truncation=True).to("cuda:0")

with torch.no_grad():
    query_embeddings = retriever_model(query=query_inputs, use_cache=False).q_reps

urls = [
    "https://huggingface.co/datasets/NTT-hil-insight/OpenDocVQA/resolve/main/image1.png",
    "https://huggingface.co/datasets/NTT-hil-insight/OpenDocVQA/resolve/main/image2.png",
]
doc_images = [prepare_doc_image(Image.open(BytesIO(requests.get(u).content))) for u in urls]

doc_embeddings = []
for img in doc_images:
    inputs = processor(DOC_PROMPT, images=img, return_tensors="pt", padding="longest", max_length=4096, truncation=True).to("cuda:0")
    with torch.no_grad():
        emb = retriever_model(document=inputs, use_cache=False).p_reps
    doc_embeddings.append(emb)
    torch.cuda.empty_cache()
doc_embeddings = torch.cat(doc_embeddings, dim=0)

for i in range(len(queries)):
    sims = cosine_similarity(query_embeddings[i].unsqueeze(0), doc_embeddings)
    print(f"Query {i} ({queries[i][:40]}...): {sims.cpu().float().numpy()}")

print("\nNTT's own reported numbers (flash_attention_2, A100, num_crops=16): [0.515625, 0.38476562] and [0.37890625, 0.5703125]")
print("CHECK 3a (retrieval): PASS if the ORDERING matches -- note we're now at num_crops=4 vs their 16, so exact values will differ more than usual.")

/content/drive/MyDrive/vdocrag-project/hf_cache/modules/transformers_modules/microsoft/Phi_hyphen_3_hyphen_vision_hyphen_128k_hyphen_instruct/ed2772fabe9dc9acd0caad54b62761d92520cc44/image_embedding_phi3_v.py:197: UserWarning: Phi-3-V modifies `input_ids` in-place and the tokens indicating images will be removed after model forward. If your workflow requires multiple forward passes on the same `input_ids`, please make a copy of `input_ids` before passing it to the model.
  warnings.warn(


Query 0 (What is the total percentage of Palestin...): [0.5234375 0.40625  ]
Query 1 (How many international visitors came to ...): [0.3984375 0.5859375]

NTT's own reported numbers (flash_attention_2, A100, num_crops=16): [0.515625, 0.38476562] and [0.37890625, 0.5703125]
CHECK 3a (retrieval): PASS if the ORDERING matches -- note we're now at num_crops=4 vs their 16, so exact values will differ more than usual.


In [10]:
from vdocrag_app.generator import build_chat_prompt

peft_model.set_adapter("generator")  # swaps the SAME peft_model both wrapper objects share

# build_chat_prompt() appends "\n Answer briefly." internally (see generator.py) --
# pass the bare question here, matching how VDocGeneratorWrapper.answer() calls it,
# rather than double-appending the suffix as a literal copy of test.py would.
prompt = build_chat_prompt(processor, "How many international visitors came to Japan in 2017?", num_images=len(doc_images))
processed = processor(prompt, images=doc_images, return_tensors="pt").to("cuda:0")

generation_args = {"max_new_tokens": 64, "temperature": 0.0, "do_sample": False, "eos_token_id": processor.tokenizer.eos_token_id}
generate_ids = generator_model.generate(processed, generation_args=generation_args, use_cache=False)
generate_ids = generate_ids[:, processed["input_ids"].shape[1]:]
response = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
print("Model prediction:", response)
print("\nNTT's own reported output (flash_attention_2, A100): '28.69m'")
print("CHECK 3b (generation): PASS if this is a sane numeric answer in the same ballpark --")
print("exact wording/formatting may differ under eager attention + 4-bit quantization.")

AttributeError: 'DynamicCache' object has no attribute 'get_usable_length'

## Results summary

Run this last -- pulls every timing/VRAM number logged above into one place, and
is exactly the data `vdocrag.limits.observed_seconds_per_page_from_log` and the
Section 4.4 VRAM-sharing decision need to move from "estimated" to "measured."

In [ ]:
import pandas as pd
df = pd.read_json('/content/drive/MyDrive/vdocrag-project/logs/step1_smoke_test.jsonl', lines=True)
display(df[df.component == 'smoke_test'][['function', 'duration_ms', 'vram_before_gb', 'vram_after_gb', 'vram_peak_gb', 'status']])